In [ ]:
!ls /kaggle/input/datasets/belhadjadji/hybrid-test/test


In [ ]:
!ls /kaggle/input/models/belhadjadji/yolo-model/other/default/1/best.pt

In [ ]:
!pip install torchmetrics segmentation-models-pytorch albumentations -q

In [ ]:
!pip install ultralytics

In [ ]:
import os, csv, time, json, cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import segmentation_models_pytorch as smp
from torchvision.ops import box_iou, nms
from ultralytics import YOLO as _YOLO
from scipy import ndimage  # used for connected component labeling on binary masks
import albumentations as A 
from albumentations.pytorch import ToTensorV2
from torchmetrics.detection import MeanAveragePrecision as _TorchMAP
from pathlib import Path

# learned-specific imports
from sklearn.linear_model import LogisticRegression

import matplotlib
matplotlib.use('Agg')

import warnings
warnings.filterwarnings("ignore")
import PIL.Image as PILImage #Used specifically in the supervisor architecture to convert numpy arrays to PIL format before applying torchvision transforms

# PATHS
YOLO_WEIGHTS      = "/kaggle/input/models/douaabel/yolo/other/default/1/best.pt"
UNET_FULL_WEIGHTS = "/kaggle/input/models/douaabel/unet-model/other/default/1/best_unet_fullimage.pth"
IMAGES_DIR = "/kaggle/input/datasets/belhadjadji/hybrid-test/test/images"
LABELS_DIR = "/kaggle/input/datasets/belhadjadji/hybrid-test/test/labels"
MASKS_DIR  = "/kaggle/input/datasets/belhadjadji/hybrid-test/test/test_masks"
BASE_OUT   = "/kaggle/working/final_results"
CKPT_FILE  = "/kaggle/working/unified_checkpoint.json"

VAL_IMAGES = "/kaggle/input/datasets/belhadjadji/hybrid-test/val/images"
VAL_LABELS = "/kaggle/input/datasets/belhadjadji/hybrid-test/val/labels"

CLASS_NAMES = ['Baton','Pliers','Hammer','Powerbank','Scissors',
               'Wrench','Gun','Bullet','Sprayer','HandCuffs','Knife','Lighter']
N_CLASSES   = len(CLASS_NAMES)

# CONSTANTS
CONF_MAP   = 0.001   # Very low confidence threshold for initial YOLO detection ( prefered high recall at detection stage later stages filter false positives)
CONF_OPER  = 0.35    # Operating threshold: only count predictions above this for Precision/Recall (why not use only CONF_OPER: because we would lose recall early and miss objects that U-Net could still recover.) 
NMS_THRESH = 0.7     # IoU threshold for NMS (high = keep more overlapping boxes)

IMG_FULL   = 512     # U-Net input size for full image
IMG_CROP   = 256     # U-Net input size for cropped region
SEG_THRESH = 0.50    # Pixel probability above this = foreground
EPS        = 1e-7    # Small number to avoid division by zero

# learned hybrid constants
learned_CROP_PAD       = 12    # Padding around detected box before cropping
learned_CROP_SIZE      = 128   # Crop resize for learned fusion
learned_QUALITY_REJECT = 0.25  # Reject crops with <25% foreground
learned_CONF_HIGH_FALLBACK = 0.70  # If conf >= this, skip U-Net (trust YOLO directly)
learned_NMS_THRESH     = 0.35  # Tighter NMS for learned arch


# Supervisor constants
SUPERVISOR_CONF_THRESH      = 0.25
SUPERVISOR_IOU_THRESH       = 0.45
SUPERVISOR_UNET_MASK_THRESH = 0.5
SUPERVISOR_OVERLAP_THRESH   = 0.15
SUPERVISOR_IMG_SIZE         = 640

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Transforms : resize, normalize, convert to tensor 
tf_full = A.Compose([
    A.Resize(IMG_FULL, IMG_FULL),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])
tf_crop = A.Compose([
    A.Resize(IMG_CROP, IMG_CROP),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])
tf_learned_crop = A.Compose([
    A.Resize(learned_CROP_SIZE, learned_CROP_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

import torchvision.transforms as tvtf
tf_supervisor = tvtf.Compose([
    tvtf.ToTensor(),
    tvtf.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),#These mean/std values are ImageNet statistics used because the U-Net encoder (ResNet34) was pretrained on ImageNet. Normalization makes the input distribution match what the encoder expects.
])

# U-NET ARCHITECTURE (Extract features twice stronger representation) 
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        # Sequential feature extractor
        self.net = nn.Sequential(

            # First convolution: extract low-level features (edges, textures)
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),

            # Normalize activations to stabilizes training
            nn.BatchNorm2d(out_ch),

            # Non-linearity to add learning capacity
            nn.ReLU(inplace=True),

            # Second convolution to refine features further
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),

            # Normalize again
            nn.BatchNorm2d(out_ch),

            # Final activation for non-linearity
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)
  
# Custom UNet used by the supervision architecture.
class UNetCustom(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512]):
        super().__init__()

        # Lists for encoder and decoder blocks
        self.downs = nn.ModuleList()   
        self.ups = nn.ModuleList()     

        # Max pooling reduces spatial size by half 
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        ch = in_channels
        # ENCODER 
        for f in features:
            # DoubleConv: feature extraction at each level
            self.downs.append(DoubleConv(ch, f))
            ch = f 
        # BOTTLENECK 
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        # DECODER 
        for f in reversed(features):
            # Transposed convolution = upsampling
            self.ups.append(nn.ConvTranspose2d(f * 2, f, kernel_size=2, stride=2))
            # After concatenation we have double the channels so we use DoubleConv to refine
            self.ups.append(DoubleConv(f * 2, f))

        # Final 1x1 convolution to produces segmentation mask
        self.final = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):

        skips = []  # store encoder outputs for skip connections

        # ENCODER FORWARD PASS
        for down in self.downs:
            x = down(x)        # feature extraction
            skips.append(x)    # save for skip connection
            x = self.pool(x)   # downsample

        # bottleneck processing
        x = self.bottleneck(x)

        # reverse skip connections for decoder alignment
        skips = skips[::-1]

        # DECODER FORWARD PASS
        for i in range(0, len(self.ups), 2):

            # Step 1: upsample feature map
            x = self.ups[i](x)

            # corresponding skip connection
            s = skips[i // 2]

            # safety check (size mismatch fix)
            if x.shape != s.shape:
                x = F.interpolate(x, size=s.shape[2:])

            # Step 2: concatenate skip + upsampled features
            x = torch.cat([s, x], dim=1)

            # Step 3: refine combined features
            x = self.ups[i + 1](x)

        # final segmentation mask output
        return self.final(x)
    
# MODEL LOADERS 
def load_yolo(path=YOLO_WEIGHTS):
    m = _YOLO(path); m.to(device) # To ensure computation runs on GPU 
    print(f"[YOLO]  {path}")
    return m

def load_unet_smp(path=UNET_FULL_WEIGHTS, label="UNet-SMP"):
    m = smp.Unet(encoder_name="resnet34", encoder_weights=None, # we use our own trained weights
                 in_channels=3, classes=1, activation=None).to(device)
    state = torch.load(path, map_location=device)
    m.load_state_dict(state); m.eval()
    print(f"[{label}]  {path}")
    return m

def load_unet_custom(path=UNET_FULL_WEIGHTS, label="UNet-Custom"):
    # Initialize model
    model = UNetCustom(in_channels=3, out_channels=1).to(device)

    # Load checkpoint
    ckpt = torch.load(path, map_location=device)
    # Extract weights
    state_dict = ckpt.get("model_state_dict", ckpt)
    # Load weights
    model.load_state_dict(state_dict, strict=False) #allows loading even if some keys are missing or extra 
    model.eval()
    print(f"[{label}] loaded from {path}")
    return model
#Why do we need to rebuild the model before loading weights: Because PyTorch only stores parameters, not the architecture itself.


# SHARED INFERENCE HELPERS
def run_yolo(model, img_bgr, conf=CONF_MAP):
    r = model.predict(img_bgr, conf=conf, iou=NMS_THRESH, verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0:
        return (torch.zeros((0, 4)), torch.zeros(0), torch.zeros(0, dtype=torch.long))
    return (r.boxes.xyxy.cpu().float(),
            r.boxes.conf.cpu().float(),
            r.boxes.cls.cpu().long())

def run_unet_full(model, img_bgr):
    H, W = img_bgr.shape[:2]
    rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    t    = tf_full(image=rgb)['image'].unsqueeze(0).float().to(device)
    with torch.no_grad():
        p = torch.sigmoid(model(t))[0, 0].cpu()
    return F.interpolate(p.unsqueeze(0).unsqueeze(0),
                         size=(H, W), mode='bilinear',
                         align_corners=False)[0, 0].numpy()

def run_unet_crop(model, img_bgr, box, pad=10):
    H_img, W_img = img_bgr.shape[:2]
    x1, y1, x2, y2 = [int(v) for v in box]
    cx1 = max(0, x1 - pad);     cy1 = max(0, y1 - pad)
    cx2 = min(W_img, x2 + pad); cy2 = min(H_img, y2 + pad)
    if cx2 <= cx1 or cy2 <= cy1: return 0.0
    crop = cv2.cvtColor(img_bgr[cy1:cy2, cx1:cx2], cv2.COLOR_BGR2RGB)
    t    = tf_crop(image=crop)['image'].unsqueeze(0).float().to(device)
    with torch.no_grad():
        prob = torch.sigmoid(model(t))[0, 0].cpu().numpy()
    ch, cw    = cy2 - cy1, cx2 - cx1
    prob_orig = cv2.resize(prob, (cw, ch))
    bx1 = max(0, x1 - cx1); by1 = max(0, y1 - cy1)
    bx2 = min(cw, x2 - cx1); by2 = min(ch, y2 - cy1)
    if bx2 <= bx1 or by2 <= by1: return 0.0
    region = prob_orig[by1:by2, bx1:bx2]
    return float((region > SEG_THRESH).sum()) / (region.size + EPS)
#Connected components :Splits segmentation mask into separate objects,Removes small noise regions,Converts each object into a bounding box
def extract_components(binary_mask, prob_full, min_area=300):
    labeled, n = ndimage.label(binary_mask)
    boxes, probs = [], []
    for cid in range(1, n + 1):
        comp = (labeled == cid)
        if comp.sum() < min_area: continue
        rows, cols = np.where(comp)
        boxes.append(torch.tensor([float(cols.min()), float(rows.min()),
                                   float(cols.max()), float(rows.max())]))
        probs.append(float(prob_full[rows, cols].mean()))
    return boxes, probs
#Ensures outputs are valid PyTorch tensors
def _safe_tensor(boxes, scores, classes):
    return (boxes.cpu().float(), scores.cpu().float(), classes.cpu().long())
#Assigns class to new U-Net detection:  If close to YOLO box → inherit YOLO class  Else → use most frequent past class
def _infer_class_for_new_det(ub, yolo_boxes, yolo_classes, cls_hist):
    if len(yolo_boxes) > 0:
        uc_x = (ub[0] + ub[2]) / 2; uc_y = (ub[1] + ub[3]) / 2
        yc_x = (yolo_boxes[:, 0] + yolo_boxes[:, 2]) / 2
        yc_y = (yolo_boxes[:, 1] + yolo_boxes[:, 3]) / 2
        dists = ((yc_x - uc_x) ** 2 + (yc_y - uc_y) ** 2).sqrt()
        nearest_idx  = dists.argmin().item()
        nearest_dist = dists[nearest_idx].item()
        if nearest_dist < 200:
            return int(yolo_classes[nearest_idx].item())
    if cls_hist: return max(cls_hist, key=cls_hist.get)
    return 0

# learned_fusion HELPERS
def _learned_crop_and_run_unet(model, img_bgr, box_xyxy):
    H, W = img_bgr.shape[:2]
    x1, y1, x2, y2 = box_xyxy
    cx1 = max(0, int(x1) - learned_CROP_PAD); cy1 = max(0, int(y1) - learned_CROP_PAD)
    cx2 = min(W, int(x2) + learned_CROP_PAD); cy2 = min(H, int(y2) + learned_CROP_PAD)
    if cx2 - cx1 < 4 or cy2 - cy1 < 4: return None, None
    crop_rgb = cv2.cvtColor(img_bgr[cy1:cy2, cx1:cx2], cv2.COLOR_BGR2RGB)
    t = tf_learned_crop(image=crop_rgb)['image'].unsqueeze(0).float().to(device)
    with torch.no_grad():
        prob = torch.sigmoid(model(t))[0, 0].cpu().numpy()
    binary = (prob > SEG_THRESH).astype(np.uint8)
    return prob, binary
#extract segmentation-based quality signals
def _learned_compute_quality_features(prob, binary):
    fg_pixels = int(binary.sum()); total = binary.size
    fg_ratio  = fg_pixels / (total + EPS)
    if fg_ratio < learned_QUALITY_REJECT: return None
    if fg_pixels > 0:
        labeled, n = ndimage.label(binary)
        sizes      = [int((labeled == i).sum()) for i in range(1, n + 1)] if n > 0 else [0]
        blob_size  = max(sizes) / (fg_pixels + EPS)
    else:
        blob_size = 0.0
    return float(fg_ratio), float(blob_size), float(prob.mean())
#Logistic Regression is lightweight, interpretable, and outputs a probability. It learns how to optimally combine YOLO confidence with 
# segmentation quality features while avoiding the complexity of training another deep network.
class LogisticFusion:
    def __init__(self):
        self.model     = LogisticRegression(max_iter=1000, C=1.0)
        self.fitted    = False
        self.weights   = None
        self.bias      = None
        self.conf_high = learned_CONF_HIGH_FALLBACK  #if calibration fails use default threshold

    def fit(self, features, labels):
        if len(features) < 20:
            print("[LogisticFusion] Not enough samples — using fallback."); return
        self.model.fit(features, labels)
        self.fitted  = True
        self.weights = self.model.coef_[0].tolist()
        self.bias    = float(self.model.intercept_[0])
        print("\n[LogisticFusion] Learned weights:")
        for n, w in zip(['yolo_conf', 'fg_ratio', 'blob_size', 'mask_mean'], self.weights):
            print(f"  {n:12s}  w = {w:+.4f}")
        print(f"  {'bias':12s}  b = {self.bias:+.4f}")

    def score(self, yolo_conf, fg_ratio, blob_size, mask_mean):
        if not self.fitted: #To provide a robust fallback when calibration data is insufficient.
            quality = 0.40 * fg_ratio + 0.35 * blob_size + 0.25 * mask_mean
            return 0.75 * yolo_conf + 0.25 * quality
        x = np.array([[yolo_conf, fg_ratio, blob_size, mask_mean]])
        return float(self.model.predict_proba(x)[0, 1])

_learned_fusion_layer = LogisticFusion()

def _iou_match(pred_box, gt_boxes, iou_thresh=0.5): #the standard threshold used to define a correct detection in object detection benchmarks.
    if len(gt_boxes) == 0: return False
    ious = box_iou(pred_box.unsqueeze(0), gt_boxes)[0]
    return bool((ious >= iou_thresh).any())





def calibrate_learned(yolo_model, unet_model,
                  val_images_dir=VAL_IMAGES, val_labels_dir=VAL_LABELS,
                  iou_thresh=0.5, n_conf_bins=20):
    print("\n[calibrate_learned] Running on validation set")
    if not os.path.isdir(val_images_dir):
        print(f"[calibrate_learned] Val dir not found . "
              f"Fallback CONF_HIGH={learned_CONF_HIGH_FALLBACK}"); return
    ext   = ('.jpg', '.jpeg', '.png', '.bmp')
    files = sorted([f for f in os.listdir(val_images_dir) if f.lower().endswith(ext)])
    if not files: print("[calibrate_learned] No val images found."); return

    features_list = []; labels_list = []
    conf_edges = np.linspace(0.0, 1.0, n_conf_bins + 1)
    bin_quality = [[] for _ in range(n_conf_bins)]
    bin_tp      = [[] for _ in range(n_conf_bins)]

    for fname in files:#Iterate over validation images.
        img_bgr = cv2.imread(os.path.join(val_images_dir, fname))
        if img_bgr is None: continue
        H, W = img_bgr.shape[:2]; stem = os.path.splitext(fname)[0]
        gt_b, gt_c = load_gt_boxes(os.path.join(val_labels_dir, stem + '.txt'), W, H)
        # Run YOLO to get predicted boxes, scores, classes
        r = yolo_model.predict(img_bgr, conf=CONF_MAP, iou=learned_NMS_THRESH, verbose=False)[0]
        if r.boxes is None or len(r.boxes) == 0: continue
        yb = r.boxes.xyxy.cpu().float(); ys = r.boxes.conf.cpu().float()
        yc = r.boxes.cls.cpu().long()
        for i in range(len(yb)):
            box  = yb[i]; conf = float(ys[i]); cls = int(yc[i])
            same = (gt_c == cls); gt_b_cls = gt_b[same] if same.any() else gt_b[:0]
            #Determine if this YOLO detection is a true positive based on IoU with ground truth boxes of the same class.
            is_tp = int(_iou_match(box, gt_b_cls, iou_thresh))
            # run unet
            prob, binary = _learned_crop_and_run_unet(unet_model, img_bgr, box.tolist())
            if prob is None: continue
            #Produces:Probability mask ,Binary mask
            feats = _learned_compute_quality_features(prob, binary)
            if feats is None: continue
            # Extract quality features
            fg_ratio, blob_size, mask_mean = feats
            quality = 0.40 * fg_ratio + 0.35 * blob_size + 0.25 * mask_mean
            #Store features and labels for logistic regression training
            features_list.append([conf, fg_ratio, blob_size, mask_mean])
            labels_list.append(is_tp)
            b_idx = min(int(conf / (1.0 / n_conf_bins)), n_conf_bins - 1)
            bin_quality[b_idx].append(quality); bin_tp[b_idx].append(is_tp)

    if not features_list: print("[calibrate_learned] No features"); return
    _learned_fusion_layer.fit(np.array(features_list), np.array(labels_list)) # Train logistic regression on collected features and labels

    conf_edges_arr = np.linspace(0.0, 1.0, n_conf_bins + 1) 
    conf_high_empirical = learned_CONF_HIGH_FALLBACK; correlations = []
    for b_idx in range(n_conf_bins):
        q_arr = np.array(bin_quality[b_idx]); t_arr = np.array(bin_tp[b_idx])
        if len(q_arr) < 5: correlations.append(np.nan); continue
        if q_arr.std() < EPS or t_arr.std() < EPS: correlations.append(0.0); continue
        correlations.append(float(np.corrcoef(q_arr, t_arr)[0, 1]))
    for b_idx in range(n_conf_bins - 1, -1, -1):
        c = correlations[b_idx]
        if np.isnan(c): continue
        if abs(c) >= 0.10: conf_high_empirical = float(conf_edges_arr[b_idx + 1]); break
    _learned_fusion_layer.conf_high = conf_high_empirical 
    print(f"[calibrate_learned] Empirical CONF_HIGH = {conf_high_empirical:.3f}") 











#Loads the ground truth bounding boxes from YOLO label files.
def load_gt_boxes(label_path, W, H):
    boxes, classes = [], []
    if not os.path.exists(label_path):
        return torch.zeros((0, 4)), torch.zeros(0, dtype=torch.long)
    with open(label_path) as f:
        for line in f:
            p = line.strip().split()
            if len(p) < 5: continue
            cls = int(p[0]); cx, cy, w, h = map(float, p[1:5])
            boxes.append([(cx - w / 2) * W, (cy - h / 2) * H,
                           (cx + w / 2) * W, (cy + h / 2) * H])
            classes.append(cls)
    if not boxes:
        return torch.zeros((0, 4)), torch.zeros(0, dtype=torch.long)
    return (torch.tensor(boxes, dtype=torch.float32),
            torch.tensor(classes, dtype=torch.long))

def load_gt_mask(mask_path, H, W):
    if not os.path.exists(mask_path): return None
    m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if m is None: return None
    if m.shape != (H, W): m = cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)
    return (m > 127).astype(np.float32)

def get_category(fname):
    n = fname.lower()
    if 'easy'   in n: return 'easy'
    if 'hard'   in n: return 'hard'
    if 'hidden' in n: return 'hidden'
    return 'other'

# Compute standard COCO detection metrics using TorchMetrics
class COCOMapMetric:
    def __init__(self):
        self._m = _TorchMAP(iou_type="bbox", class_metrics=True)

    def update(self, pb, ps, pc, gb, gc):
        pb, ps, pc = _safe_tensor(pb, ps, pc)
        gb, _, gc  = _safe_tensor(gb, torch.zeros(len(gc)), gc)
        self._m.update(
            [{'boxes': pb, 'scores': ps, 'labels': pc.int()}],
            [{'boxes': gb, 'labels': gc.int()}]
        )

    def compute(self):
        r   = self._m.compute()
        out = {
            'mAP@0.5':      max(0., float(r['map_50'].item())),
            'mAP@0.5:0.95': max(0., float(r['map'].item())),
            'mAR@100':      max(0., float(r.get('mar_100', torch.tensor(-1.)).item())),
        }
        if 'map_per_class' in r:
            for i, v in enumerate(r['map_per_class'].tolist()):
                nm = CLASS_NAMES[i] if i < N_CLASSES else f'cls{i}'
                out[f'mAP50_{nm}'] = max(0., float(v))
        return out

    def reset(self): self._m.reset()


class OperatingPointMetric:
    def __init__(self, iou_thresh=0.5):
        self.iou_t = iou_thresh; self.reset()

    def reset(self): self.tp = self.fp = self.fn = 0

    def update(self, pb, ps, pc, gb, gc):
        keep = ps >= CONF_OPER
        pb, ps, pc = pb[keep], ps[keep], pc[keep]
        if len(pb) == 0: self.fn += len(gb); return
        if len(gb) == 0: self.fp += len(pb); return
        order  = ps.argsort(descending=True)
        pb, pc = pb[order], pc[order]
        iou_mat = box_iou(pb, gb); matched = set()
        for i in range(len(pb)):
            bv, bj = -1, -1
            for j in range(len(gb)):
                if j in matched or pc[i] != gc[j]: continue
                if iou_mat[i, j] > bv: bv, bj = iou_mat[i, j].item(), j
            if bv >= self.iou_t: matched.add(bj); self.tp += 1
            else: self.fp += 1
        self.fn += len(gb) - len(matched)

    def compute(self):
        p  = (self.tp + EPS) / (self.tp + self.fp + EPS)
        r  = (self.tp + EPS) / (self.tp + self.fn + EPS)
        f1 = 2 * p * r / (p + r + EPS)
        return {'Precision': p, 'Recall': r, 'F1': f1}

# CHECKPOINT
def save_checkpoint(results):
    with open(CKPT_FILE, 'w') as f: json.dump(results, f, indent=2)
    print(f"  Checkpoint → {CKPT_FILE}")

def load_checkpoint():
    if os.path.exists(CKPT_FILE):
        with open(CKPT_FILE) as f: data = json.load(f)
        print(f"  Resuming from checkpoint: {list(data.keys())}"); return data
    return {}

# EVALUATION LOOP
def run_evaluation(fusion_fn, output_dir, arch_name, save_vis=True, n_vis=15):
    os.makedirs(output_dir, exist_ok=True)
    if save_vis: os.makedirs(os.path.join(output_dir, 'vis'), exist_ok=True)

    ext   = ('.jpg', '.jpeg', '.png', '.bmp')
    files = sorted([f for f in os.listdir(IMAGES_DIR) if f.lower().endswith(ext)])
    print(f"\n[{arch_name}]  {len(files)} images")

    map_metric  = COCOMapMetric()
    oper_metric = OperatingPointMetric()
    per_rows    = []; vis_count = 0; t0 = time.time()

    for idx, fname in enumerate(files):
        img_bgr = cv2.imread(os.path.join(IMAGES_DIR, fname))
        if img_bgr is None: continue
        H, W  = img_bgr.shape[:2]; stem = os.path.splitext(fname)[0]
        gt_b, gt_c = load_gt_boxes(os.path.join(LABELS_DIR, stem + '.txt'), W, H)

        fb, fs, fc, pred_mask = fusion_fn(img_bgr)

        map_metric.update(fb, fs, fc, gt_b, gt_c)
        oper_metric.update(fb, fs, fc, gt_b, gt_c)

        row = {'image': fname, 'n_gt': len(gt_b), 'n_pred': len(fb)}
        per_rows.append(row)

        if save_vis and vis_count < n_vis:
            _save_vis(img_bgr, gt_b, fb, fs, fc, pred_mask,
                      os.path.join(output_dir, 'vis', fname))
            vis_count += 1

        if (idx + 1) % 2000 == 0:
            elapsed = (time.time() - t0) / 60; fps = (idx + 1) / (time.time() - t0)
            print(f"  {idx+1}/{len(files)}  {elapsed:.1f}min  {fps:.1f}fps")

    det_map  = map_metric.compute()
    det_oper = oper_metric.compute()
    all_m = {}
    all_m['Precision']      = det_oper['Precision']
    all_m['Recall']         = det_oper['Recall']
    all_m['mAP@0.5']        = det_map['mAP@0.5']
    all_m['mAP@0.5:0.95']   = det_map['mAP@0.5:0.95']
    all_m['F1']             = det_oper['F1']
    all_m['mAR@100']        = det_map['mAR@100']
    # Per-class mAP
    for k, v in det_map.items():
        if k.startswith('mAP50_'):
            all_m[k] = v

    elapsed = (time.time() - t0) / 60
    print(f"\n{'='*60}")
    print(f"  {arch_name}  ({elapsed:.1f} min)")
    print(f"{'='*60}")
    for k in ['Precision', 'Recall', 'mAP@0.5', 'F1']:
        v = all_m.get(k, float('nan'))
        vstr = f"{v:.4f}" if not np.isnan(v) else "N/A"
        print(f"  {k:<25} {vstr:>10}")

    # Save CSVs
    with open(os.path.join(output_dir, 'summary_metrics.csv'), 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(all_m.keys()))
        w.writeheader()
        w.writerow({k: f"{v:.4f}" if not np.isnan(v) else 'nan' for k, v in all_m.items()})
    if per_rows:
        with open(os.path.join(output_dir, 'per_image.csv'), 'w', newline='') as f:
            w = csv.DictWriter(f, fieldnames=per_rows[0].keys())
            w.writeheader(); w.writerows(per_rows)
    print(f"  Saved → {output_dir}")
    return all_m

def _save_vis(img_bgr, gt_b, fb, fs, fc, pred_mask, out_path):
    vis = img_bgr.copy()
    if pred_mask is not None:
        hot = (pred_mask > SEG_THRESH).astype(np.uint8)
        vis[hot == 1] = (vis[hot == 1] * 0.6 + np.array([0, 180, 0]) * 0.4).astype(np.uint8)
    for box in gt_b:
        x1, y1, x2, y2 = box.int().tolist()
        cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 255, 0), 2)
    for box, score, cls in zip(fb, fs, fc):
        if score < CONF_OPER: continue
        x1, y1, x2, y2 = box.int().tolist()
        col = (0, 220, 0) if score > 0.5 else (0, 140, 255)
        cv2.rectangle(vis, (x1, y1), (x2, y2), col, 2)
        nm = CLASS_NAMES[int(cls)] if int(cls) < N_CLASSES else '?'
        cv2.putText(vis, f"{nm} {score:.2f}", (x1, max(y1 - 4, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.35, col, 1)
    cv2.imwrite(out_path, vis)


#   yolo_baseline 
def make_arch_yolo_only(yolo_model):
    def fusion(img_bgr):
        yb, ys, yc = run_yolo(yolo_model, img_bgr)
        return yb, ys, yc, None
    return fusion

# ARCHITECTURE 1 : arch1_score_seq 
def make_arch1(yolo_model, unet_model):

    HIGH_CONF    = 0.50; CONF_BOOST = 0.06; CONF_PENALTY = 0.18

    def fusion(img_bgr):
        yb, ys, yc = run_yolo(yolo_model, img_bgr)
        fb, fs, fc = [], [], []
        for i in range(len(yb)):
            box = yb[i]; score = ys[i].item(); cls = yc[i]
            if score >= HIGH_CONF:
                fb.append(box); fs.append(torch.tensor(score)); fc.append(cls); continue
            ratio = run_unet_crop(unet_model, img_bgr, box.tolist())
            if ratio >= 0.30: score = min(1., score + CONF_BOOST * ratio)
            else:             score = max(0., score - CONF_PENALTY * (1 - ratio))
            fb.append(box); fs.append(torch.tensor(score)); fc.append(cls)
        if not fb:
            return (torch.zeros((0, 4)), torch.zeros(0), torch.zeros(0, dtype=torch.long), None)
        return torch.stack(fb), torch.stack(fs), torch.stack(fc), None

    return fusion

# ARCHITECTURE 2 : supervisor_hybrid  
def make_supervisor_hybrid(yolo_model, unet_custom_model):
    #run supervisor U-Net to get binary mask of potential object regions
    def _run_unet_supervisor(img_bgr):
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_rs  = cv2.resize(img_rgb, (SUPERVISOR_IMG_SIZE, SUPERVISOR_IMG_SIZE))
        img_pil = PILImage.fromarray(img_rs)
        img_t   = tf_supervisor(img_pil)
        with torch.no_grad():
            logit = unet_custom_model(img_t.unsqueeze(0).to(device))
            prob  = torch.sigmoid(logit)[0, 0].cpu().numpy()
        return (prob > SUPERVISOR_UNET_MASK_THRESH).astype(np.uint8)

    def _hybrid_filter(yolo_preds_raw, unet_mask, orig_hw):
        H, W = orig_hw; mH, mW = unet_mask.shape
        filtered_boxes, filtered_scores, filtered_classes = [], [], []
        for pb, ps, pc in zip(*yolo_preds_raw):
            x1, y1, x2, y2 = pb.tolist()
            mx1=int(x1/W*mW); my1=int(y1/H*mH)
            mx2=int(x2/W*mW); my2=int(y2/H*mH)
            mx1,my1=max(0,mx1),max(0,my1); mx2,my2=min(mW,mx2),min(mH,my2)
            if mx2<=mx1 or my2<=my1: continue
            roi     = unet_mask[my1:my2, mx1:mx2]
            overlap = roi.sum() / max(roi.size, 1)
            if overlap >= SUPERVISOR_OVERLAP_THRESH:
                filtered_boxes.append(pb); filtered_scores.append(ps)
                filtered_classes.append(pc)
        return filtered_boxes, filtered_scores, filtered_classes

    def fusion(img_bgr):
        H, W = img_bgr.shape[:2]
       
        r = yolo_model.predict(img_bgr, conf=SUPERVISOR_CONF_THRESH,
                               iou=SUPERVISOR_IOU_THRESH, verbose=False)[0]
        if r.boxes is None or len(r.boxes) == 0:
            return (torch.zeros((0,4)), torch.zeros(0), torch.zeros(0,dtype=torch.long), None)
        yb = r.boxes.xyxy.cpu().float()
        ys = r.boxes.conf.cpu().float()
        yc = r.boxes.cls.cpu().long()

        unet_mask = _run_unet_supervisor(img_bgr)

        pred_mask = cv2.resize(unet_mask.astype(np.float32),
                               (W, H), interpolation=cv2.INTER_NEAREST)

        fboxes, fscores, fclasses = _hybrid_filter((yb, ys, yc), unet_mask, (H, W))

        if not fboxes:
            return (torch.zeros((0,4)), torch.zeros(0), torch.zeros(0,dtype=torch.long), pred_mask)
        return (torch.stack(fboxes),
                torch.stack(fscores),
                torch.stack(fclasses),
                pred_mask)

    return fusion

# ARCHITECTURE 3 : arch_learned 
def make_arch_learned(yolo_model, unet_model):
    conf_high = _learned_fusion_layer.conf_high

    def fusion(img_bgr):
        r = yolo_model.predict(img_bgr, conf=CONF_MAP, iou=learned_NMS_THRESH, verbose=False)[0]
        if r.boxes is None or len(r.boxes) == 0:
            return (torch.zeros((0, 4)), torch.zeros(0), torch.zeros(0, dtype=torch.long), None)
        yb = r.boxes.xyxy.cpu().float(); ys = r.boxes.conf.cpu().float()
        yc = r.boxes.cls.cpu().long()
        out_boxes, out_scores, out_classes = [], [], []
        for i in range(len(yb)):
            box  = yb[i]; conf = float(ys[i]); cls = yc[i]
            if conf >= conf_high:
                out_boxes.append(box); out_scores.append(torch.tensor(conf))
                out_classes.append(cls); continue
            prob, binary = _learned_crop_and_run_unet(unet_model, img_bgr, box.tolist())
            if prob is None:
                out_boxes.append(box); out_scores.append(torch.tensor(conf))
                out_classes.append(cls); continue
            feats = _learned_compute_quality_features(prob, binary)
            if feats is None: continue
            fg_ratio, blob_size, mask_mean = feats
            S_final = _learned_fusion_layer.score(conf, fg_ratio, blob_size, mask_mean)
            if S_final < CONF_OPER: continue
            out_boxes.append(box); out_scores.append(torch.tensor(float(S_final)))
            out_classes.append(cls)
        if not out_boxes:
            return (torch.zeros((0, 4)), torch.zeros(0), torch.zeros(0, dtype=torch.long), None)
        boxes_t   = torch.stack(out_boxes)
        scores_t  = torch.tensor([s.item() for s in out_scores])
        classes_t = torch.stack(out_classes)
        keep_idx  = []
        for c in classes_t.unique():
            mask = classes_t == c; idx = torch.where(mask)[0]
            kept = nms(boxes_t[idx], scores_t[idx], learned_NMS_THRESH)
            keep_idx.extend(idx[kept].tolist())
        keep_idx = sorted(keep_idx)
        return (boxes_t[keep_idx], scores_t[keep_idx], classes_t[keep_idx], None)

    return fusion

# ARCHITECTURE 4 : arch_parallel
def make_arch_parallel(yolo_model, unet_model):

    OVERLAP_CONFIRM = 0.40;  BOOST = 0.30; PEN = 0.15

    def fusion(img_bgr):
        H, W       = img_bgr.shape[:2]
        yb, ys, yc = run_yolo(yolo_model, img_bgr)
        prob       = run_unet_full(unet_model, img_bgr)
        pred_mask  = (prob > SEG_THRESH).astype(np.float32)
        fb, fs, fc = [], [], []
        for i in range(len(yb)):
            box = yb[i]; score = ys[i].item(); cls = yc[i]
            x1, y1, x2, y2 = [int(v) for v in box.tolist()]
            x1=max(0,x1); y1=max(0,y1); x2=min(W,x2); y2=min(H,y2)
            if x2 > x1 and y2 > y1:
                ov = float((prob[y1:y2, x1:x2] > SEG_THRESH).mean())
                if ov >= OVERLAP_CONFIRM: score = min(1., score + BOOST * ov)
                elif ov < OVERLAP_CONFIRM:   score = max(0., score - PEN * (1 - ov))
            fb.append(box); fs.append(torch.tensor(score)); fc.append(cls)
        if not fb:
            return (torch.zeros((0,4)), torch.zeros(0), torch.zeros(0,dtype=torch.long), pred_mask)
        return torch.stack(fb), torch.stack(fs), torch.stack(fc), pred_mask

    return fusion

# ARCHITECTURE 5 : arch5_attention
def make_arch5_attention(yolo_model, unet_model):
    ALPHA_K     = 8.0
    ALPHA_PIVOT = 0.50
    ALPHA_MIN = 0.55
    SCORE_FLOOR = 0.88
    IOU_MATCH   = 0.30
    W_PROB = 0.50
    W_IOU = 0.50
    NEW_CONF_MIN = 0.82
    MAX_NEW = 3
    MIN_COMP = 400
    IOU_EXISTING = 0.20
    MIN_CENTRE_DIST = 150

    def _sig(x): return max(ALPHA_MIN, 1. / (1. + np.exp(-x)))

    def fusion(img_bgr):
        H, W = img_bgr.shape[:2]
        yb, ys, yc             = run_yolo(yolo_model, img_bgr)
        prob                   = run_unet_full(unet_model, img_bgr)
        binary                 = (prob > SEG_THRESH).astype(np.uint8)
        unet_boxes, unet_probs = extract_components(binary, prob, MIN_COMP)
        pred_mask              = binary.astype(np.float32)
        cls_hist = {}
        for c in yc.tolist(): cls_hist[int(c)] = cls_hist.get(int(c), 0) + 1
        yolo_centres = []
        if len(yb) > 0:
            yolo_centres = torch.stack([(yb[:, 0]+yb[:, 2])/2, (yb[:, 1]+yb[:, 3])/2], dim=1)
        stage1_boxes, stage1_scores, stage1_classes = [], [], []
        used = set()
        for i in range(len(yb)):
            box = yb[i]; score = ys[i].item(); cls = yc[i]
            x1, y1, x2, y2 = [int(v) for v in box.tolist()]
            x1=max(0,x1); y1=max(0,y1); x2=min(W,x2); y2=min(H,y2)
            mean_prob = float(prob[y1:y2, x1:x2].mean()) if x2>x1 and y2>y1 else 0.
            alpha = _sig(ALPHA_K * (score - ALPHA_PIVOT))
            iou_support = 0.
            if unet_boxes:
                ut   = torch.stack(unet_boxes); ious = box_iou(box.unsqueeze(0), ut)[0]
                bv, bj = ious.max().item(), ious.argmax().item()
                if bv >= IOU_MATCH: iou_support = bv; used.add(bj)
            unet_signal = W_PROB * mean_prob + W_IOU * iou_support
            blended = float(np.clip(alpha * score + (1 - alpha) * unet_signal, 0., 1.))
            blended = max(blended, score * SCORE_FLOOR)
            if score >= CONF_OPER and blended < CONF_OPER:
                blended = max(CONF_OPER + 0.01, score * 0.95)
            stage1_boxes.append(box); stage1_scores.append(blended); stage1_classes.append(cls)
        if stage1_boxes:
            s1b = torch.stack(stage1_boxes); s1s = torch.tensor(stage1_scores)
            s1c = torch.stack(stage1_classes); keep_idx = []
            for c in s1c.unique():
                mask = s1c == c; idx = torch.where(mask)[0]
                kept = nms(s1b[idx], s1s[idx], NMS_THRESH); keep_idx.extend(idx[kept].tolist())
            keep_idx = sorted(keep_idx)
            s1b=s1b[keep_idx]; s1s=s1s[keep_idx]; s1c=s1c[keep_idx]
        else:
            s1b=torch.zeros((0,4)); s1s=torch.zeros(0); s1c=torch.zeros(0,dtype=torch.long)
        new_boxes, new_scores, new_classes = [], [], []; new_count = 0
        for j, (ub, cp) in enumerate(zip(unet_boxes, unet_probs)):
            if j in used or cp < NEW_CONF_MIN or new_count >= MAX_NEW: continue
            if len(yolo_centres) > 0:
                ub_cx=(ub[0]+ub[2])/2; ub_cy=(ub[1]+ub[3])/2
                dists=((yolo_centres[:,0]-ub_cx)**2+(yolo_centres[:,1]-ub_cy)**2).sqrt()
                if dists.min().item() < MIN_CENTRE_DIST: continue
            if len(s1b) > 0:
                if box_iou(ub.unsqueeze(0), s1b)[0].max().item() > IOU_EXISTING: continue
            assigned_cls = _infer_class_for_new_det(ub, yb, yc, cls_hist)
            if assigned_cls not in cls_hist: continue
            new_boxes.append(ub); new_scores.append(float(np.clip(cp*0.75,0.,1.)))
            new_classes.append(torch.tensor(assigned_cls, dtype=torch.long)); new_count += 1
        all_boxes = list(s1b) + new_boxes
        all_scores = list(s1s.numpy()) + new_scores
        all_cls    = list(s1c) + new_classes
        if not all_boxes:
            return (torch.zeros((0,4)), torch.zeros(0), torch.zeros(0,dtype=torch.long), pred_mask)
        return (torch.stack(all_boxes), torch.tensor(all_scores, dtype=torch.float32),
                torch.stack(all_cls), pred_mask)

    return fusion

# ARCHITECTURE 6 : arch6_attention
def make_arch6_attention(yolo_model, unet_model):

    #  Attention fusion 
    ALPHA_K       = 6.0
    ALPHA_PIVOT   = 0.40
    ALPHA_MIN     = 0.55

    #  Score stabilization 
    SCORE_FLOOR   = 0.92

    #  U-Net support 
    IOU_MATCH     = 0.25
    W_PROB = 0.40
    W_IOU  = 0.60
    MIN_COMP      = 200

    #  Controlled recall recovery 
    RESCUE_IOU   = 0.30
    RESCUE_CONF  = 0.60
    RESCUE_BOOST = 0.18

    def _sig(x):
        return max(ALPHA_MIN, 1. / (1. + np.exp(-x)))

    def fusion(img_bgr):

        H, W = img_bgr.shape[:2]

        #  YOLO detections 
        yb, ys, yc = run_yolo(yolo_model, img_bgr)

        #  Full-image U-Net prediction 
        prob = run_unet_full(unet_model, img_bgr)

        binary = (prob > SEG_THRESH).astype(np.uint8)

        #  Extract connected components 
        unet_boxes, _ = extract_components(binary, prob, MIN_COMP)

        pred_mask = binary.astype(np.float32)

        if len(yb) == 0:
            return (
                torch.zeros((0,4)),
                torch.zeros(0),
                torch.zeros(0,dtype=torch.long),
                pred_mask
            )

        blended_boxes   = []
        blended_scores  = []
        blended_classes = []

        #  Fusion process 
        for i in range(len(yb)):

            box   = yb[i]
            score = ys[i].item()
            cls   = yc[i]

            x1, y1, x2, y2 = [int(v) for v in box.tolist()]

            x1 = max(0, x1)
            y1 = max(0, y1)
            x2 = min(W, x2)
            y2 = min(H, y2)

            #  Mean probability inside box 
            if x2 > x1 and y2 > y1:
                mean_prob = float(prob[y1:y2, x1:x2].mean())
            else:
                mean_prob = 0.0

            #  Adaptive attention weight 
            alpha = _sig(ALPHA_K * (score - ALPHA_PIVOT))

            #  IoU support 
            iou_support = 0.0

            if unet_boxes:

                ut = torch.stack(unet_boxes)

                ious = box_iou(box.unsqueeze(0), ut)[0]

                best_iou = ious.max().item()

                if best_iou >= IOU_MATCH:
                    iou_support = best_iou

            #  U-Net signal 
            unet_signal = (
                W_PROB * mean_prob +
                W_IOU  * iou_support
            )

            #  Controlled rescue mechanism 
            if (
                iou_support > RESCUE_IOU and
                score < RESCUE_CONF
            ):

                score = min(
                    1.0,
                    score + RESCUE_BOOST * iou_support
                )

            #  Final confidence fusion 
            blended = float(np.clip(
                alpha * score +
                (1 - alpha) * unet_signal,
                0.0,
                1.0
            ))

            #  Prevent excessive degradation 
            blended = max(
                blended,
                score * SCORE_FLOOR
            )

            #  Preserve valid YOLO detections 
            if score >= CONF_OPER and blended < CONF_OPER:

                blended = max(
                    CONF_OPER + 0.02,
                    score * 0.95
                )

            blended_boxes.append(box)
            blended_scores.append(blended)
            blended_classes.append(cls)

        #  Convert to tensors 
        b_t = torch.stack(blended_boxes)

        s_t = torch.tensor(
            blended_scores,
            dtype=torch.float32
        )

        c_t = torch.stack(blended_classes)

        #  Per-class NMS 
        keep_idx = []

        for c in c_t.unique():

            mask = c_t == c

            idx = torch.where(mask)[0]

            kept = nms(
                b_t[idx],
                s_t[idx],
                NMS_THRESH
            )

            keep_idx.extend(idx[kept].tolist())

        keep_idx = sorted(keep_idx)

        return (
            b_t[keep_idx],
            s_t[keep_idx],
            c_t[keep_idx],
            pred_mask
        )

    return fusion  

# DISPLAY ORDER FOR FINAL TABLE
DISPLAY_ORDER = [
    "yolo_baseline",
    "arch1_score_seq",
    "supervisor_hybrid", 
    "arch_learned",
    "arch_parallel",
    "arch5_attention",
    "arch6_attention"
]

# FINAL SUMMARY TABLE
def print_final_table(all_results):
    print(f"\n{'='*85}")
    print(f"FINAL COMPARISON")
    print(f"{'='*85}")
    hdr = (f"{'Architecture':28s} {'Precision':>10} {'Recall':>10} "
           f"{'mAP@0.5':>10} {'F1':>10}")
    print(hdr); print('-' * 85)
    yolo_m50 = all_results.get('yolo_baseline', {}).get('mAP@0.5', 0)
    for nm in DISPLAY_ORDER:
        if nm not in all_results: continue
        r = all_results[nm]
        p    = r.get('Precision', 0); rc = r.get('Recall', 0)
        m50  = r.get('mAP@0.5', float('nan')); f1 = r.get('F1', 0)
        m50s = f"{m50:.4f}" if not np.isnan(m50) else "   N/A  "
        flag = ''
        if nm != 'yolo_baseline' and not np.isnan(m50) and m50 > yolo_m50:
            flag = ' '
        print(f"  {nm:26s}  {p:>10.4f}  {rc:>10.4f}  {m50s:>10}  {f1:>10.4f}{flag}")
    print(f"\n    beats YOLO baseline mAP@0.5 ({yolo_m50:.4f})")

# MAIN
print("Loading models")
yolo_model      = load_yolo()
unet_smp_model  = load_unet_smp()         
unet_cust_model = load_unet_custom()       
os.makedirs(BASE_OUT, exist_ok=True)

#  Step 0: Calibrate learned fusion on validation split 
calibrate_learned(yolo_model, unet_smp_model)

#  Step 1: Define architectures 
archs = [
    ("yolo_baseline",
     make_arch_yolo_only(yolo_model)),

    ("arch1_score_seq",
     make_arch1(yolo_model, unet_smp_model)),
   
    ("supervisor_hybrid",
     make_supervisor_hybrid(yolo_model, unet_cust_model)),

    ("arch_learned",
     make_arch_learned(yolo_model, unet_smp_model)),

    ("arch_parallel",
     make_arch_parallel(yolo_model, unet_smp_model)),    
  
    ("arch5_attention",
     make_arch5_attention(yolo_model, unet_smp_model)),
  
    ("arch6_attention",
     make_arch6_attention(yolo_model, unet_smp_model)),
]

#  Step 2: Run evaluations (checkpoint-resumable) 
all_results = load_checkpoint()

for name, fn in archs:
    if name in all_results:
        print(f"\nSKIPPING {name} (already in checkpoint)")
        continue
    print(f"\n{'='*60}\nRUNNING: {name}\n{'='*60}")
    result = run_evaluation(fn, os.path.join(BASE_OUT, name), name)
    all_results[name] = result
    save_checkpoint(all_results)

#  Step 3: Print final table 
print_final_table(all_results)

#  Step 4: Save comparison CSV 
fields = ['arch', 'Precision', 'Recall', 'mAP@0.5', 'F1',
          'mAP@0.5:0.95', 'mAR@100']
with open(os.path.join(BASE_OUT, 'comparison_all.csv'), 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fields); w.writeheader()
    for nm in DISPLAY_ORDER:
        if nm not in all_results: continue
        r = all_results[nm]
        w.writerow({'arch': nm,
                    **{k: f"{r.get(k, float('nan')):.4f}"
                       for k in fields[1:]}})

print(f"\nAll results saved → {BASE_OUT}/")